# 04 · Análisis de Componentes Principales (PCA) de clientes

**Objetivo:** resumir las 7 variables numéricas de los clientes en pocos componentes y entender qué las
explica (edad, vuelos, gasto, gasto extra, millas, ingreso, anticipación de compra).

**Datos:** `Clientes_Demorados_SinDuplicados.csv`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from vuelos_ds import config, data, dimred, eda, plots

plots.set_style()
import numpy as np

## 1. Datos y correlaciones

In [ ]:
clientes = data.load_clientes_demorados()
features = config.CLIENTES_NUMERICAS
print("Variables:", features)

plots.plot_matrix(clientes, features, "corr")
plt.show()
display(eda.correlaciones_altas(clientes[features], umbral=0.45))

## 2. Test de esfericidad de Bartlett

Verifica que las variables estén suficientemente correlacionadas para que PCA tenga sentido.
H₀: la matriz de correlación es la identidad. Si p < 0,05 se rechaza H₀.

In [ ]:
pca, X, x_pca = dimred.fit_pca(clientes, features)

chi2, p = dimred.bartlett(X)
print(f"Chi-cuadrado de Bartlett: {chi2:.2f} | p-value: {p:.4f}")
print("Se rechaza H0: hay correlación suficiente para PCA." if p < 0.05
      else "No se rechaza H0: las variables no están suficientemente correlacionadas.")

## 3. Varianza explicada

In [ ]:
display(dimred.tabla_varianza(pca).round(3))
print(f"Componentes con autovalor > 1 (Kaiser): {dimred.kaiser(pca)}")
plots.plot_scree(pca)
plt.show()

## 4. Loadings: qué variables componen cada componente

In [ ]:
cargas = dimred.loadings(pca, features)
display(cargas.round(3))

plt.figure(figsize=(9, 4))
sns.heatmap(cargas, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Loadings por componente")
plt.show()

## 5. Datos proyectados

In [ ]:
plots.plot_pc_pairs(x_pca, [(0, 1), (0, 2), (1, 2), (0, 3), (1, 3), (2, 3)])
plt.show()

## 6. Conclusiones

*(Valores de la corrida original con los datos reales.)*

- **PC1 (44,1 % de la varianza)** agrupa gasto acumulado, gasto extra, cantidad de vuelos y millas (loadings
  0,41–0,52): es un eje de **intensidad de uso / valor del cliente**.
- **PC2 (15,1 %)** contrasta **edad** (+0,71) con **anticipación de compra** (−0,68).
- **PC3 (13,2 %)** combina edad y anticipación en el mismo sentido (+0,67 y +0,72), y **PC4 (11,1 %)** queda
  dominado por el **ingreso mensual** (0,92).
- **Cuántos componentes quedarse:** por Kaiser (autovalor > 1) alcanzan 2 (59 % de la varianza); con 4 se explica
  el 84 %.
- El último componente (autovalor ≈ 0,09) opone gasto acumulado y gasto extra: refleja la **colinealidad** entre
  ambas variables.